# Calculating the residual CE left after accounting for the model.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

if not os.path.exists('data'):
    os.mkdir('data')

# path to save data to on completion
VIS_PATH = os.path.join('data', 'web_vis')
if not os.path.exists(VIS_PATH):
    os.mkdir(VIS_PATH)

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join('data', 'final-document.tsv')

In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

['../data/lme_ready/cha-ceda-analysis.tsv',
 '../data/lme_ready/candor-ceda-analysis.tsv',
 '../data/lme_ready/cha-CABNC-ceda-analysis.tsv',
 '../data/lme_ready/grace-ceda-analysis.tsv',
 '../data/lme_ready/cha-MICASE-ceda-analysis.tsv',
 '../data/lme_ready/cha-coraal-ceda-analysis.tsv']

Iteratively import dataframes and editing them

In [5]:
start_from_turn = 5

In [6]:
df,keep_cols = [], ['speaker', 'in_common', 'our_thoughts_synced_up_sr1', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5', 'saw_world_in_same_way_sr8', 'responsive_1', 'responsive_2', 'responsive_3', 'conv_leader']
keep_cols = ['x_'+col for col in keep_cols] + ['conversation_id', 'corpus', 'dyad', 'self', 'turn_delta', 'nx', 'ny', 'Hxy']
len(keep_cols)

18

In [7]:
for f in tqdm(fs):
    df += [pd.read_table(f, sep='\t')]
    
    if ('/cha-' in f) or ('grace-' in f):
        df[-1]['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df[-1]['x_speaker'])]
   
    elif '/candor-' in f:
        df[-1]['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df[-1]['conversation_id'])]
        df[-1]['corpus'] = 'CANDOR'
    
    else:
        if 'conversation_id' in df[-1].columns:
            0
        else:
            df[-1]['conversation_id'] = f
        
    df[-1] = df[-1].loc[
        (df[-1]['nx'] >= 5) 
        & (df[-1]['ny'] >= 5) 
        & (df[-1]['x_turn_id'] >= start_from_turn)
    ]
    
    df[-1]['dyad'] = df[-1]['x_speaker'] + '-' + df[-1]['y_speaker']
    df[-1]['turn_delta'] = (df[-1]['y_turn_id'] - df[-1]['x_turn_id'])
    df[-1] = df[-1][[col for col in list(df[-1]) if col in keep_cols]]
    

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/8028150 [00:00<?, ?it/s]

  0%|          | 0/429480 [00:00<?, ?it/s]

  0%|          | 0/12723777 [00:00<?, ?it/s]

  0%|          | 0/695015 [00:00<?, ?it/s]

  0%|          | 0/8262622 [00:00<?, ?it/s]

  0%|          | 0/11936993 [00:00<?, ?it/s]

In [9]:
df = pd.concat(df, ignore_index=True)
df.shape

(22954906, 14)

In [10]:
df['conversation_id'].nunique()

2634

In [11]:
df.head()

,x_speaker,corpus,conversation_id,self,nx,ny,Hxy,dyad,turn_delta,x_in_common,x_our_thoughts_synced_up_sr1,x_developed_joint_perspective_sr2,x_thoughts_became_more_alike_sr5,x_saw_world_in_same_way_sr8
0,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,False,20.0,6.0,6.344819,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,1,NaN,NaN,NaN,NaN,NaN
1,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,False,20.0,8.0,6.000488,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,2,NaN,NaN,NaN,NaN,NaN
2,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,False,20.0,13.0,5.521488,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,4,NaN,NaN,NaN,NaN,NaN
3,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,False,20.0,6.0,6.057978,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,6,NaN,NaN,NaN,NaN,NaN
4,GCSAusE-GCSAusE-01-S,GCSAusE,GCSAusE-GCSAusE-01,False,20.0,6.0,6.198739,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,7,NaN,NaN,NaN,NaN,NaN


## Convert values to numeric tags

In [ ]:
convert_to_numeric_id = [
    'x_speaker', 'y_speaker', 
    'dyad', 
    #'x_speaker_turn'
]

In [ ]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col] = [conversion[v] for v in tqdm(df[col])]
    

## Save a checkpoint

In [ ]:
# df[['Hxy', 'dyad', 'turn_delta', 'self', 'corpus', 'conversation_id']].to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')

## LME Regression: Predicting CE

In [ ]:
from tqdm import tqdm
import statsmodels.formula.api as smf

In [ ]:
##########################################
## Main model
##########################################
model = "Hxy ~ nx + ny + turn_delta*self"
##########################################

In [ ]:
start = dt.now()
# md = smf.mixedlm(model, data=df_, groups=df_['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['dyad'])
mdf = md.fit()
print('completed in:', dt.now()-start)

In [ ]:
# mdf.summary()

In [ ]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

Saving results for reporting

In [ ]:
df['resid'] = mdf.resid

In [ ]:
sel = df['corpus'].apply(lambda x: 'candor' in str(x).lower())
sel.sum()

In [ ]:
df.loc[sel].to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')

In [ ]:
# df.loc[sel].head()